In [1]:
# fix_wbt_race.py
import re

files_and_vars = [
    ("whitebox-tools-app/src/tools/hydro_analysis/sink.rs", "filled_dem"),
    ("whitebox-tools-app/src/tools/hydro_analysis/depth_in_sink.rs", "filled_dem"),
    ("whitebox-tools-app/src/tools/hydro_analysis/fill_depressions.rs", "output"),
]

for filepath, varname in files_and_vars:
    with open(filepath, "r") as f:
        lines = f.readlines()

    out = []
    i = 0
    inserted_handles_decl = False
    inserted_join = False

    while i < len(lines):
        line = lines[i]

        # 1. Insert handles declaration before "for tid in 0..num_procs {"
        if not inserted_handles_decl and re.match(r"^\s*for tid in 0\.\.num_procs \{", line):
            indent = re.match(r"^(\s*)", line).group(1)
            out.append(f"{indent}let mut handles = Vec::with_capacity(num_procs as usize);\n")
            inserted_handles_decl = True
            out.append(line)
            i += 1
            continue

        # 2. Replace "thread::spawn(move || {" with "handles.push(thread::spawn(move || {"
        if re.match(r"^\s*thread::spawn\(move \|\| \{", line):
            out.append(line.replace("thread::spawn(move || {", "handles.push(thread::spawn(move || {"))
            i += 1
            # 3. Find the matching closing "});" at the same indent as thread::spawn and change to "}));"
            spawn_indent = re.match(r"^(\s*)", line).group(1)
            depth = 1
            while i < len(lines):
                l2 = lines[i]
                if l2.strip() == "});" and l2.startswith(spawn_indent) and len(re.match(r"^(\s*)", l2).group(1)) == len(spawn_indent):
                    out.append(l2.replace("});", "}));"))
                    i += 1
                    break
                else:
                    out.append(l2)
                    i += 1
            continue

        # 4. Insert join loop before "VAR = match Arc::try_unwrap(VAR2) {"
        pattern = rf"^\s*{varname} = match Arc::try_unwrap\({varname}2\) \{{"
        if not inserted_join and re.match(pattern, line):
            indent = re.match(r"^(\s*)", line).group(1)
            out.append(f"{indent}// Ensure all spawned threads (and their Arc clones) have fully\n")
            out.append(f"{indent}// exited before Arc::try_unwrap -- fixes intermittent\n")
            out.append(f"{indent}// \"Error unwrapping '{varname}'\" panics caused by a race condition.\n")
            out.append(f"{indent}for h in handles {{\n")
            out.append(f"{indent}    h.join().expect(\"Thread panicked\");\n")
            out.append(f"{indent}}}\n")
            inserted_join = True
            out.append(line)
            i += 1
            continue

        out.append(line)
        i += 1

    if not (inserted_handles_decl and inserted_join):
        print(f"WARNING: {filepath} - handles_decl={inserted_handles_decl}, join={inserted_join} (pattern may not match, check manually!)")
    else:
        print(f"OK: {filepath} patched successfully")

    with open(filepath, "w") as f:
        f.writelines(out)

OK: whitebox-tools-app/src/tools/hydro_analysis/sink.rs patched successfully
OK: whitebox-tools-app/src/tools/hydro_analysis/depth_in_sink.rs patched successfully
OK: whitebox-tools-app/src/tools/hydro_analysis/fill_depressions.rs patched successfully
